In [17]:
"Load Data"

import re

def read_text_file(file_path: str) -> list[str]:
    """
    读取本地英文文本文件：
    1. 剔除所有非英文字母字符，统一替换为空格
    2. 去除首尾空白、全部转为小写
    3. 自动过滤清洗后的空行
    Args:
        file_path: 本地文本文件的完整路径
    Returns:
        list[str]: 每行清洗后的文本组成的列表
    """
    # 打开文件：指定utf-8编码，遇异常字符自动替换，避免乱码报错
    with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
        lines = f.readlines()

    # 逐行清洗，额外过滤空行
    cleaned_lines = []
    for line in lines:
        # 正则：所有非英文字母的连续字符，替换为单个空格
        line_processed = re.sub('[^A-Za-z]+', ' ', line)
        # 去首尾空格 + 全小写
        line_processed = line_processed.strip().lower()
        # 过滤空行（原d2l未处理，属于优化点）
        if line_processed:
            cleaned_lines.append(line_processed)

    return cleaned_lines


In [23]:
"将文本序列拆成 token"

def tokenize(lines, token='word'):
    if token == 'word':
        return [line.split() for line in lines]
    elif token == 'char':
        return [list(line) for line in lines]
    else:
        print('unknowed token:' + token)
        

In [19]:
"""
整理词表，将 token 映射为数字索引
将出现频度很小的 token 从语料库 corpus 中剔除，存放到另一个库中
"""

import collections
from collections import Counter

def count_corpus(tokens):
    "统计语料词频"

    # 空输入直接返回空计数器
    if len(tokens) == 0:
        return Counter()
    # 判断是二维嵌套列表（每行分词结果），展平为一维
    if isinstance(tokens[0], list):
        tokens = [token for line in tokens for token in line]
    return Counter(tokens)


class Vocab:
    
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
        # 空输入默认初始化为空列表
        if tokens is None:
            tokens = []
        if reserved_tokens is None:
            reserved_tokens = []

        # 1. 统计全部词元频率，并按频率从高到低排序
        counter = count_corpus(tokens)
        # 按词频降序排列 (token, freq)
        self._token_freqs = sorted(counter.items(), key=lambda x: x[1], reverse=True)

        # 2. 初始化索引-词双向映射
        # 首位固定<unk>未知词，再拼接用户自定义保留词
        self.idx_to_token = ['<unk>'] + reserved_tokens
        self.token_to_idx = {token: idx for idx, token in enumerate(self.idx_to_token)}

        # 3. 过滤低频词，构建完整词表映射
        for token, freq in self._token_freqs:
            # 低于最小频率阈值直接跳过，后续全部不再遍历
            if freq < min_freq:
                break
            # 避免和保留token重复
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, tokens):

        # 单个词元输入
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.unk)
        # 列表/元组批量转换
        return [self.__getitem__(token) for token in tokens]

    def to_tokens(self, indices):
        
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[idx] for idx in indices]

    @property
    def unk(self):
        "未知词元<unk>固定索引为0"
        return 0

    @property
    def token_freqs(self):
        return self._token_freqs

In [20]:
file_path = r'C:\Users\17934\Desktop\MachineLearning\Program\TextData.txt'

def load_text_data(max_tokens=-1):

    """返回数据集的词元索引列表和词表"""
    
    lines = read_text_file(file_path)
    
    tokens = tokenize(lines, 'char')

    vocab = Vocab(tokens)
    
    corpus = [vocab[token] for line in tokens for token in line]

    if max_tokens > 0:
        corpus = corpus[:max_tokens]

    return corpus, vocab